In [24]:
import pennylane as qml
from pennylane import numpy as np


In [25]:
from sklearn.datasets import load_iris

data = load_iris()
X = data.data
y = data.target

print(X.shape)  # (150, 4)

(150, 4)


In [26]:
import random

X_min = X.min(axis=0)   # min of each column
X_max = X.max(axis=0)   # max of each column

X = (X - X_min) / (X_max - X_min)

X = X * (2*np.pi) - np.pi   # [-π, π]



# Example lists (columns)
A = X
B = y

# Combine into rows
rows = list(zip(A, B))

# Shuffle rows
random.shuffle(rows)

# Unzip back into columns
A_shuf, B_shuf = zip(*rows)

# (optional) convert back to lists
X= list(A_shuf)
y = list(B_shuf)



X_train = X[:100]
y_train = y[:100]

X_test = X[100:]
y_test = y[100:]


In [28]:
dev = qml.device("default.qubit", wires = 3)

In [27]:
@qml.qnode(dev)
def circuit(params, x):
    
    qml.RX(x[0], 0)
    qml.RX(x[1], 1)
    qml.RX(x[2], 2)

    qml.RY(params[0], 0)
    qml.RY(params[1], 1)
    qml.RY(params[2], 2)
    qml.CNOT([2, 1])
    #qml.Hadamard(0)
    #qml.CNOT([1, 0])
    qml.RZ(params[3], 0)
    qml.RZ(params[4], 1)
    qml.RZ(params[5], 2)


    #qml.CNOT([0, 2])


    qml.RX(x[1], 0)    
    qml.RX(x[2], 1)
    qml.RX(x[3], 2)

    qml.RY(params[6], 0)
    qml.RY(params[7], 1)
    qml.RY(params[8], 2)
    #qml.CNOT([2, 1])
    qml.CNOT([1, 0])
    #qml.Hadamard(2)
    qml.RZ(params[9], 0)
    qml.RZ(params[10], 1)
    qml.RZ(params[11], 2)


    """qml.CNOT([0, 2])


    qml.RX(x[0] + x[1], 0)
    qml.RX(x[1] + x[2], 1)
    qml.RX(x[2] + x[3], 2)"""
    qml.RX(x[2], 0)
    qml.RX(x[3], 1)
    qml.RX(x[0], 2)    

    qml.CNOT([0, 1])
    qml.CNOT([1, 2])


    qml.RY(params[12], 0)
    qml.RY(params[13], 1)
    qml.RY(params[14], 2)
    qml.CNOT([0, 2])
    qml.RZ(params[15], 0)
    qml.RZ(params[16], 1)
    qml.RZ(params[17], 2)


    """qml.CNOT([0, 2])


    qml.RX(x[2] + x[3], 0)
    qml.RX(x[3] + x[0], 1)
    qml.RX(x[0] + x[1], 2)"""


        
    return qml.probs(wires = [0, 1, 2])

In [29]:
def cost(params, X, y):
    loss = 0

    for x, y_true in zip(X, y):
        probs = circuit(params, x)

        # group states
        p0 = probs[0] * probs[1]
        p1 = probs[2] * probs[3]
        p2 = probs[4] * probs[5]

        p_vec = np.array([p0, p1, p2])

        # --- normalize (no power sharpening) ---
        p_vec = p_vec / np.sum(p_vec)

        p_true = p_vec[y_true]

        # --- margin loss (KEY) ---
        margin = 0.4
        for i in range(3):
            if i != y_true:
                loss += max(0, margin - (p_true - p_vec[i]))

        # --- small cross entropy (secondary) ---
        loss -= 0.5 * np.log(p_true + 1e-10)

        # --- unused penalty (keep moderate) ---
        unused = probs[6] + probs[7]
        loss += 0.4 * unused

    return loss / len(X)

In [ ]:
probs = circuit(params, X[0])

# group states
p0 = probs[0] * probs[1]
p1 = probs[2] * probs[3]
p2 = probs[4] * probs[5]

p_vec = np.array([p0, p1, p2])

# --- normalize (no power sharpening) ---
p_vec = p_vec / np.sum(p_vec)


In [30]:
def compute_accuracy(params, X, y):
    correct = 0

    for x, y_true in zip(X, y):
        probs = circuit(params, x)

        p0 = probs[0] * probs[1]
        p1 = probs[2] * probs[3]
        p2 = probs[4] * probs[5]

        y_pred = np.argmax([p0, p1, p2])

        if y_pred == y_true:
            correct += 1

    return correct / len(X)

In [34]:
import random

n = 120

np.random.seed(n)
random.seed(n)   # <-- this was missing

params = np.random.rand(18, requires_grad=True) #* (2*np.pi) - np.pi   # [-π, π]

In [35]:
opt = qml.AdamOptimizer(stepsize=0.05)

steps = 50

for i in range(steps):
    if i > 30:
        opt = qml.AdamOptimizer(stepsize=0.01)
    params = opt.step(lambda p: cost(p, X_train, y_train), params)

    if i % 2 == 0:
        train_loss = cost(params, X_train, y_train)
        train_acc = compute_accuracy(params, X_train, y_train)
        test_acc = compute_accuracy(params, X_test, y_test)

        print(f"Step {i:3d} | Loss: {train_loss:.6f} | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")

Step   0 | Loss: 1.438319 | Train Acc: 0.4700 | Test Acc: 0.3600
Step   2 | Loss: 1.110020 | Train Acc: 0.6000 | Test Acc: 0.4800
Step   4 | Loss: 0.877324 | Train Acc: 0.7200 | Test Acc: 0.7400
Step   6 | Loss: 0.736245 | Train Acc: 0.7400 | Test Acc: 0.8400
Step   8 | Loss: 0.648109 | Train Acc: 0.7700 | Test Acc: 0.8800
Step  10 | Loss: 0.583607 | Train Acc: 0.7900 | Test Acc: 0.8800
Step  12 | Loss: 0.569891 | Train Acc: 0.8400 | Test Acc: 0.8800
Step  14 | Loss: 0.568553 | Train Acc: 0.8300 | Test Acc: 0.8600
Step  16 | Loss: 0.557745 | Train Acc: 0.8300 | Test Acc: 0.8800
Step  18 | Loss: 0.545632 | Train Acc: 0.8500 | Test Acc: 0.8800
Step  20 | Loss: 0.524441 | Train Acc: 0.8500 | Test Acc: 0.8800
Step  22 | Loss: 0.498214 | Train Acc: 0.8600 | Test Acc: 0.8600
Step  24 | Loss: 0.469712 | Train Acc: 0.8600 | Test Acc: 0.8800
Step  26 | Loss: 0.448678 | Train Acc: 0.8600 | Test Acc: 0.8800
Step  28 | Loss: 0.433124 | Train Acc: 0.8700 | Test Acc: 0.8800
Step  30 | Loss: 0.421855

In [36]:
unused_vals = []

for x in X_test:
    probs = circuit(params, x)
    unused_vals.append(probs[6] + probs[7])

print("Avg unused:", np.mean(unused_vals))

Avg unused: 0.18767595561487535


In [37]:
vals = []
for x in X_test:
    probs = circuit(params, x)
    p0 = probs[0] + probs[1]
    p1 = probs[2] + probs[3]
    p2 = probs[4] + probs[5]
    #print( probs[0], probs[1], probs[2], probs[3], probs[4], probs[5],probs[6], probs[7])
    confidence = max(p0, p1, p2)
    vals.append(confidence)

print("Avg confidence:", np.mean(vals))

Avg confidence: 0.5002171079856045
